In [1]:
import pandas as pd
import numpy as np

# Load raw data
df = pd.read_parquet('../data/raw/yellow_tripdata_2024-01.parquet')
zones = pd.read_csv('../data/raw/taxi_zone_lookup.csv')
df_clean = df.copy()

print(f"StaStarting records: 2,964,624
=======================================================rting records: {len(df_clean):,}")
print("=" * 55)

Starting records: 2,964,624


In [2]:
# ============================================
# STEP 1: Filter to January 2024 only
# ============================================
before = len(df_clean)
df_clean = df_clean[
    (df_clean['tpep_pickup_datetime'] >= '2024-01-01') &
    (df_clean['tpep_pickup_datetime'] < '2024-02-01') &
    (df_clean['tpep_dropoff_datetime'] >= '2024-01-01') &
    (df_clean['tpep_dropoff_datetime'] < '2024-02-01')
]
after = len(df_clean)
print(f"STEP 1 - Date filter: Removed {before - after:,} rows")
print(f"  Remaining: {after:,}")

STEP 1 - Date filter: Removed 618 rows
  Remaining: 2,964,006


In [3]:
# ============================================
# STEP 2: Remove duplicates
# ============================================
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
after = len(df_clean)
print(f"STEP 2 - Duplicates: Removed {before - after:,} rows")
print(f"  Remaining: {after:,}")

STEP 2 - Duplicates: Removed 0 rows
  Remaining: 2,964,006


In [4]:
# ============================================
# STEP 3: Remove invalid trip distances
# ============================================
before = len(df_clean)
df_clean = df_clean[
    (df_clean['trip_distance'] > 0) &
    (df_clean['trip_distance'] <= 100)
]
after = len(df_clean)
print(f"STEP 3 - Invalid distance: Removed {before - after:,} rows")
print(f"  Remaining: {after:,}")

STEP 3 - Invalid distance: Removed 60,426 rows
  Remaining: 2,903,580


In [5]:
# ============================================
# STEP 4: Remove invalid fares
# ============================================
before = len(df_clean)
df_clean = df_clean[
    (df_clean['fare_amount'] > 0) &
    (df_clean['fare_amount'] <= 500)
]
after = len(df_clean)
print(f"STEP 4 - Invalid fare: Removed {before - after:,} rows")
print(f"  Remaining: {after:,}")

STEP 4 - Invalid fare: Removed 34,551 rows
  Remaining: 2,869,029


In [6]:
# ============================================
# STEP 5: Fix passenger count
# ============================================
df_clean['passenger_count'] = df_clean['passenger_count'].fillna(1)
before = len(df_clean)
df_clean = df_clean[
    (df_clean['passenger_count'] >= 1) &
    (df_clean['passenger_count'] <= 6)
]
after = len(df_clean)
print(f"STEP 5 - Passenger count: Removed {before - after:,} rows")
print(f"  Remaining: {after:,}")

STEP 5 - Passenger count: Removed 30,697 rows
  Remaining: 2,838,332


In [7]:
# ============================================
# STEP 6: Calculate trip duration & filter
# ============================================
df_clean['trip_duration_min'] = (
    df_clean['tpep_dropoff_datetime'] - df_clean['tpep_pickup_datetime']
).dt.total_seconds() / 60

before = len(df_clean)
df_clean = df_clean[
    (df_clean['trip_duration_min'] > 0) &
    (df_clean['trip_duration_min'] <= 180)
]
after = len(df_clean)
print(f"STEP 6 - Invalid duration: Removed {before - after:,} rows")
print(f"  Remaining: {after:,}")

STEP 6 - Invalid duration: Removed 1,986 rows
  Remaining: 2,836,346


In [8]:
# ============================================
# STEP 7: Validate zone IDs
# ============================================
valid_zone_ids = set(zones['LocationID'])
before = len(df_clean)
df_clean = df_clean[
    (df_clean['PULocationID'].isin(valid_zone_ids)) &
    (df_clean['DOLocationID'].isin(valid_zone_ids))
]
after = len(df_clean)
print(f"STEP 7 - Invalid zones: Removed {before - after:,} rows")
print(f"  Remaining: {after:,}")

STEP 7 - Invalid zones: Removed 0 rows
  Remaining: 2,836,346


In [9]:
# ============================================
# STEP 8: Remove outliers (IQR method)
# ============================================
def remove_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    before = len(data)
    data = data[(data[column] >= lower) & (data[column] <= upper)]
    after = len(data)
    print(f"  {column}: Removed {before - after:,} (kept [{lower:.2f}, {upper:.2f}])")
    return data

print("STEP 8 - Outlier removal:")
for col in ['trip_distance', 'fare_amount', 'trip_duration_min']:
    df_clean = remove_outliers_iqr(df_clean, col)

print(f"\n  Remaining: {len(df_clean):,}")

STEP 8 - Outlier removal:
  trip_distance: Removed 365,533 (kept [-2.23, 6.45])
  fare_amount: Removed 83,299 (kept [-2.95, 27.85])
  trip_duration_min: Removed 35,436 (kept [-5.82, 27.44])

  Remaining: 2,352,078


In [10]:
# ============================================
# STEP 9: Save cleaned data
# ============================================
import os
os.makedirs('../data/processed', exist_ok=True)

df_clean.to_parquet('../data/processed/yellow_tripdata_2024-01_clean.parquet', index=False)

print("=" * 55)
print(f"FINAL CLEAN RECORDS: {len(df_clean):,}")
print(f"Records removed: {len(df) - len(df_clean):,}")
print(f"Retention rate: {len(df_clean)/len(df)*100:.1f}%")
print("=" * 55)
print("\nCleaned data saved to:")
print("  data/processed/yellow_tripdata_2024-01_clean.parquet")

FINAL CLEAN RECORDS: 2,352,078
Records removed: 612,546
Retention rate: 79.3%

Cleaned data saved to:
  data/processed/yellow_tripdata_2024-01_clean.parquet
